In [1]:
import sys
sys.path.insert(0, '..')

from src import validate_smiles, parse_smiles
import pandas as pd
from rdkit import Chem
from glob import glob

Parser debugging for SmilesParser written to parser.out


In [2]:
def validate_rdkit(mol:str) -> bool:
    new_mol = Chem.MolFromSmiles(mol)
    if new_mol is None:
        return False
    
    return True

In [3]:
def validate_rdkit(mol: str) -> bool:
    """Validate SMILES using RDKit."""
    new_mol = Chem.MolFromSmiles(mol)
    return new_mol is not None

def benchmark(use_only_grammar=False):
    """Benchmark validator against RDKit on all CSV files in data/."""
    files = glob('../data/*.csv')
    results = []
    
    for csv in files:
        try:
            df = pd.read_csv(csv)
        except UnicodeDecodeError:
            df = pd.read_csv(csv, compression='gzip')
        
        if 'smiles' not in df.columns:
            print(f"Skipping {csv} - no 'smiles' column")
            continue
            
        mols = [row['smiles'] for _, row in df.iterrows()]
        validated_by_rdkit = [validate_rdkit(mol) for mol in mols]
        
        if use_only_grammar:
            # Use parse_smiles for syntax-only validation
            validated_by_yacc = [parse_smiles(mol)[0] for mol in mols]
        else:
            # Use validate_smiles for full validation
            validated_by_yacc = [validate_smiles(mol)[0] for mol in mols]
        
        results += list(zip(mols, validated_by_rdkit, validated_by_yacc))
        print(f"Processed {csv}: {len(mols)} molecules")
    
    return results

In [4]:
# Run benchmarks
print("Running with full chemistry validation...")
bench_with_chem = benchmark(use_only_grammar=False)
print("\nRunning with syntax-only validation...")
bench_with_only_grammar = benchmark(use_only_grammar=True)

Running with full chemistry validation...


[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors
[14:41:52] WARNING: not removing hydrogen atom without neighbors


Processed ../data/sider.csv: 1427 molecules
Processed ../data/mutagenicity.csv: 4337 molecules


[14:41:55] WARNING: not removing hydrogen atom without neighbors
[14:41:55] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:41:55] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:41:55] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:41:55] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:41:56] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:41:56] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:41:56] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:41:56] Explicit valence for atom # 20 Al, 6, is greater than permitted


Processed ../data/tox21.csv: 7831 molecules
Processed ../data/muv.csv: 93087 molecules


...

Processed ../data/sider.csv: 1427 molecules
Processed ../data/mutagenicity.csv: 4337 molecules


[14:43:03] WARNING: not removing hydrogen atom without neighbors
[14:43:03] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:43:03] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:43:03] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:43:03] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:43:04] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:43:04] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:43:04] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:43:04] Explicit valence for atom # 20 Al, 6, is greater than permitted


Processed ../data/tox21.csv: 7831 molecules
Processed ../data/muv.csv: 93087 molecules


[14:44:07] Explicit valence for atom # 0 N, 4, is greater than permitted
[14:44:07] Can't kekulize mol.  Unkekulized atoms: 9
[14:44:07] Can't kekulize mol.  Unkekulized atoms: 4
[14:44:07] Can't kekulize mol.  Unkekulized atoms: 4


Processed ../data/clintox.csv: 1484 molecules


In [5]:
# Syntax-only validation results
yacc_syntax_failures = sum(1 for m, r, y in bench_with_only_grammar if not y)
rdkit_failures = sum(1 for m, r, y in bench_with_only_grammar if not r)
total = len(bench_with_only_grammar)

print("=== Syntax-Only Validation Results ===")
print(f"Total molecules: {total}")
print(f"YACC syntax failures: {yacc_syntax_failures} ({100*yacc_syntax_failures/total:.2f}%)")
print(f"RDKit failures: {rdkit_failures} ({100*rdkit_failures/total:.2f}%)")

=== Syntax-Only Validation Results ===
Total molecules: 108166
YACC syntax failures: 715 (0.66%)
RDKit failures: 12 (0.01%)


In [6]:
# Show SMILES that RDKit rejects but YACC accepts (syntax-only)
print("SMILES where RDKit fails but YACC syntax passes:")
rdkit_fails_yacc_passes = [mol for mol, rdkit, yacc in bench_with_only_grammar if not rdkit and yacc]
for mol in rdkit_fails_yacc_passes:
    print(f"  {mol[:80]}")

SMILES where RDKit fails but YACC syntax passes:
  NC(=O)NC1N=C(O[AlH3](O)O)NC1=O
  O=CO[AlH3](OC=O)OC=O
  CC(=O)O[AlH3](O)O
  CC(=O)O[AlH3](O)OC(C)=O
  CCOC(=O)/C=C(/C)O[AlH3](OC(C)CC)OC(C)CC
  CCCCO[AlH3](OCCCC)OCCCC
  CCCCCCCCCCCCCCCCCC(=O)O[AlH3](O)O
  [NH4][Pt]([NH4])(Cl)Cl
  c1ccc(cc1)n2c(=O)c(c(=O)n2c3ccccc3)CCS(=O)c4ccccc4
  CCCCc1c(=O)n(n(c1=O)c2ccc(cc2)O)c3ccccc3
  CCCCc1c(=O)n(n(c1=O)c2ccccc2)c3ccccc3


In [7]:
# Full chemistry validation results
yacc_chem_failures = sum(1 for m, r, y in bench_with_chem if not y)
total = len(bench_with_chem)

print("=== Full Chemistry Validation Results ===")
print(f"Total molecules: {total}")
print(f"YACC failures: {yacc_chem_failures} ({100*yacc_chem_failures/total:.2f}%)")
print(f"RDKit failures: {rdkit_failures} ({100*rdkit_failures/total:.2f}%)")

# Agreement analysis
both_valid = sum(1 for m, r, y in bench_with_chem if r and y)
both_invalid = sum(1 for m, r, y in bench_with_chem if not r and not y)
yacc_only_invalid = sum(1 for m, r, y in bench_with_chem if r and not y)
rdkit_only_invalid = sum(1 for m, r, y in bench_with_chem if not r and y)

print(f"\nAgreement analysis:")
print(f"  Both valid: {both_valid}")
print(f"  Both invalid: {both_invalid}")
print(f"  YACC invalid, RDKit valid: {yacc_only_invalid}")
print(f"  YACC valid, RDKit invalid: {rdkit_only_invalid}")
print(f"  Agreement rate: {100*(both_valid + both_invalid)/total:.2f}%")

=== Full Chemistry Validation Results ===
Total molecules: 108166
YACC failures: 16872 (15.60%)
RDKit failures: 12 (0.01%)

Agreement analysis:
  Both valid: 91288
  Both invalid: 6
  YACC invalid, RDKit valid: 16866
  YACC valid, RDKit invalid: 6
  Agreement rate: 84.40%


In [8]:
# Analyze failure reasons
from collections import Counter

failure_reasons = []
for mol, rdkit, yacc in bench_with_chem:
    if rdkit and not yacc:
        _, err = validate_smiles(mol)
        if err:
            # Extract the rule name from the error
            err_str = str(err)
            if 'validate_smiles' in err_str:
                failure_reasons.append('trailing_C_check')
            elif 'validate_rings_and_valency' in err_str:
                failure_reasons.append('valency_check')
            elif 'validate_aromaticity' in err_str:
                failure_reasons.append('aromaticity_check')
            elif 'chain' in err_str:
                failure_reasons.append('parsing_error')
            else:
                failure_reasons.append('other')

print("=== Failure Reason Breakdown ===")
for reason, count in Counter(failure_reasons).most_common():
    print(f"  {reason}: {count} ({100*count/len(failure_reasons):.1f}%)")

=== Failure Reason Breakdown ===
  trailing_C_check: 10390 (61.6%)
  aromaticity_check: 4368 (25.9%)
  valency_check: 1404 (8.3%)
  parsing_error: 702 (4.2%)
  other: 2 (0.0%)
